# Feature Engineering on High Volume For-Hire Vehicle (HVFHV) Trip Records Dataset:

In this notebook, we are mainly focusing on adding helpful columns in HVFHV dataset.

----

# Import Libraries:

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import * 
from pyspark.sql.functions import when, col, date_format, \
                                    unix_timestamp, to_date, hour, month
import os

In [ ]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("feature_engineering_hvfhv")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.network.timeout", "600s")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

# Read Files:

In [ ]:
base_dir = "../data"

In [ ]:
hvfhv_path = base_dir + '/curated/hvfhv_data/preprocessed_hvfhv_2'
hvfhv_sdf = spark.read.parquet(hvfhv_path)
hvfhv_sdf.show(5)

In [ ]:
num_rows = hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

# Add New Columns:

Since as a driver, they concentrate on where I go could increase the revenue, so we are focusing on the pickup location.

Add new column `day_of_week` to indicate it is Monday or Tuesday and so on based on `pickup_datetime`:

In [ ]:
hvfhv_sdf = hvfhv_sdf.withColumn("day_of_week", date_format("pickup_datetime", "EEEE"))
hvfhv_sdf.show(5)

Add new column `request_to_pickup_time` to indicate the waiting time in minutes for passengers after they request a HVFHV (calculated by `pickup_datetime` - `request_datetime`):

In [ ]:
hvfhv_sdf = hvfhv_sdf.withColumn(
    "request_to_pickup_minutes",
    (unix_timestamp("pickup_datetime") - unix_timestamp("request_datetime")) / 60
)
hvfhv_sdf.show(5)

Add new column `trip_speed` (unit: miles per minutes) calculated by `trip_miles`/`trip_time` :

In [ ]:
hvfhv_sdf = hvfhv_sdf.withColumn(
    "trip_speed",
    col("trip_miles") / col("trip_time")
)
hvfhv_sdf.show(5)

Add new column `total_fare_amount` calculated by `base_passenger_fare` + `tolls` + `bcf` + `sales_tax` + `congestion_surcharge` + `airport_fee` + `tips`:

In [ ]:
hvfhv_sdf = hvfhv_sdf.withColumn(
    "total_fare_amount",
    col("base_passenger_fare") + col("tolls") + col("bcf") + col("sales_tax") + 
    col("congestion_surcharge") + col("airport_fee") + col("tips")
)
hvfhv_sdf.show(5)

Add new column `total_revenue` calculated by `driver_pay` + `tips`:

In [ ]:
hvfhv_sdf = hvfhv_sdf.withColumn(
    "total_revenue", col("driver_pay") + col("tips")
)
hvfhv_sdf.show(5)

Extract the date and hour from the `pickup_datetime` and `dropoff_datetime` columns:


In [ ]:
hvfhv_sdf = hvfhv_sdf.withColumn("pickup_date", to_date("pickup_datetime")) \
                     .withColumn("pickup_hour", hour("pickup_datetime")) \
                     .withColumn("dropoff_date", to_date("dropoff_datetime")) \
                     .withColumn("dropoff_hour", hour("dropoff_datetime")) \
                     .withColumn("month", month("pickup_datetime"))

hvfhv_sdf = hvfhv_sdf.drop("request_datetime", "pickup_datetime", "dropoff_datetime")
hvfhv_sdf.show(5)

In [ ]:
# Check the shape the parquet file
num_rows = hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

# Save the Full HVFHV Dataset:

In [ ]:
hvfhv_dir = base_dir + '/developed/merged_data'
file_name = 'full_hvfhv'
hvfhv_path = os.path.join(hvfhv_dir, file_name)
hvfhv_sdf.write.mode('overwrite').parquet(hvfhv_path)